# Heritabiltiy from family data. 
This notebook follows the GCTA-based practical exercise with R, as described in the provided instructions.

 In this exercise you will be carrying out association analysis of data from a mini genome-wide association study. The data comes from families (related individuals) measured for a quantitative trait of interest. The purpose is detect which (if any) of the loci are associated with the quantitative trait. 

 We will be using family data consisting of 498 individuals typed at 134,946 SNPs. All individuals have measurements of a quantitative trait of interest. You can assume that appropriate quality control (QC) checks on SNPs and individuals have been carried out prior to the current analysis i.e. the data set is already QC-ed. 


## setup environment

In [ ]:
# shared tools and data folder
ROOT_PATH=/course/chinacourse2026/shared
TOOL_PATH=${ROOT_PATH}/Software
SHARED_PATH=${ROOT_PATH}/ref # For reference database
INPUT_PATH=${ROOT_PATH}/data/heritability  # for input data
ls ${INPUT_PATH}

In [ ]:
rm -rf ~/sysu_day5_heritabiltiy
mkdir -p ~/sysu_day5_heritabiltiy
cd ~/sysu_day5_heritabiltiy

cp -sf ${INPUT_PATH}/quantfam.zip .
cp -sf ${SHARED_PATH}/online.R .
cp -sf ${SHARED_PATH}/newPlotPlink.R .

unzip -o quantfam.zip

echo -----files in folder -----

ls


In [ ]:
cp -sf ${SHARED_PATH}/newPlotPlink.R .
ls

In [ ]:
# set up R working space
work_d <- path.expand("~/sysu_day5_heritabiltiy")
setwd(work_d)

source("./online.R")

In [ ]:
# set up python working space
import os
work_d = os.path.expanduser("~/sysu_day5_heritabiltiy")
os.chdir(work_d)

You should find 3 PLINK binary-format files in your directory: quantfamdata.bed, quantfamdata.bim and quantfamdata.fam. The file quantfamdata.bed is the binary genotype file which will not be human readable. The file quantfamdata.bim is a map file. You can take a look at this (e.g. by typing more quantfamdata.bim). The file quantfamdata.fam gives the pedigree structure in a format that is compatible with the binary genotype file. You can take a look at this (e.g. by typing more quantfamdata.fam). Note this file is the same as the first six columns of a standard pedigree file, with the last column giving each individual's quantitative trait value.



## Step 1: Create phenotype file in R

To start with, we will use R to create the phenotype file required by GCTA.

In [ ]:
fam <- read.table("./quantfamdata.fam", header=FALSE)
pheno <- data.frame(fam[,1:2], fam[,6])
write.table(pheno, file="./myphenos.txt", col.names=FALSE, row.names=FALSE, quote=FALSE)

cat("\nfam file\n")
head(fam)

cat("\nphnotype file\n")
head(pheno)


## Step 2: GCTA Analysis
Perform association analysis accounting for relatedness using GCTA.

In [ ]:
# 1.5 min to run
gcta64 --mlma --bfile quantfamdata --pheno myphenos.txt --out GCTAresults

### Load and visualize results in R

In [ ]:

res <- read.table("./GCTAresults.mlma", header=TRUE)
head(res)



plot

In [ ]:
manPlot(res$p,chr=res$Chr,)

In [ ]:
qqp(res$p)

chi <- qchisq(res$p, 1,lower.tail=FALSE)
lambda <- median(chi) / qchisq(0.5,1)
cat("\nInflation factor (lamdda)=",lambda)

### Estimate SNP Heritability with GCTA


To use GCTA to estimate the heritability accounted for by all autosomal
genome-wide SNPs, you need to first estimate the GRM, and then use the
GRM to estimate the (SNP) heritability. This can be achieved using the
following commands:

In [ ]:
gcta64 --bfile quantfamdata --autosome --make-grm-bin --out GCTAgrm


Identify (genetically) unrelated individuals

In [ ]:
gcta64 --grm GCTAgrm --grm-singleton 0.05 --out unrelated

Questions: 
    1) How many individuals are unrelated in this sample? GRM cut-off 0.05? cut-off 0.025?
    
    
    
   #### estimate heritabilty

In [ ]:
gcta64 --reml --grm-bin GCTAgrm --pheno myphenos.txt --out GCTAherit


**The screen output estimates the SNP heritability V(G)/Vp to be?**









# LD socre regression by LDSC

In this tutorial is based on https://cloufield.github.io/GWASTutorial/08_LDSC/. We will use sample summary statistics for HDLC and 
LDLC from Japan Biobank (JBB). 
- Kanai, Masahiro, et al. "Genetic analysis of quantitative traits in 
the Japanese population links cell types to complex human diseases." 
Nature genetics 50.3 (2018): 390-400.

### LDSC: LD Score Regression

LDSC is one of the most commonly used command-line tools to estimate:

- **Inflation** ($\lambda$) in test statistics  
- **SNP-heritability**  
- **Genetic correlation** between traits  
- **Enrichment** in specific cell or tissue types

Unlike GCTA is it based on **GWAS summary statistics** and not individual-level genotype data.

---

#### Linkage Disequilibrium (LD)

LDSC is based on **Linkage disequilibrium (LD)** which is the non-random association of alleles at different loci in a given population.  
> In other words, if you know the allele at SNP A, you can better predict the allele at SNP B.

LD is typically quantified by the squared correlation coefficient between SNPs ($r_{jk}^2$) for SNP j and SNP k, or the covariance ($D_{jk}$).

---

####  LD Score

The **LD score** of a SNP $j$ is defined as:

$$
\text{LD Score}_j = \sum_{k} r_{jk}^2
$$

Where:
- $r_{jk}^2$ is the squared correlation (LD) between SNP $j$ and SNP $k$
- The sum is typically over all SNPs within a 1 cM or 1 Mb window of SNP $j$

**Interpretation:** A SNP that is in high LD with many other SNPs (i.e., tags more variants) has a higher LD score.

---

####  LD Score Regression

**Key Idea:** A SNP will tend to have a higher test statistic if it is in LD with one or more causal variants.  
The expected chi-squared statistic at SNP $j$, assuming polygenicity, follows:

$$
\mathbb{E}[\chi_j^2] = 1 + \frac{N h^2}{M} \cdot \text{LD Score}_j + a
$$

Where:
- $\chi_j^2$: chi-squared statistic for SNP $j$
- $N$: GWAS sample size
- $M$: total number of SNPs analyzed
- $h^2$: SNP heritability (on the observed scale)
- $a$: contribution from confounding bias (e.g., cryptic relatedness, population stratification)

The above formula can be rewritten as

$$\chi_j^2 = \alpha + \beta x + \epsilon$$
where $x=\frac{N \cdot \text{LD Score}_j}{M} $, $\alpha=1+a$ and $\beta=h^2$

**This is the key regression equation used in LDSC:**
- Regress $\chi^2$ on LD score
- The **slope** gives $\frac{N h^2}{M}$
- The **intercept** estimates inflation ($a$) due to confounding plus 1. If the intercept is close to 1 then it means that there is no confounding and thus any inflation of p-values is due to the trait being highly polygenic



## Setup the environment

In [ ]:
INPUT_PATH=${ROOT_PATH}/data/ldsc  # for input data
LDSC_ENV="/home/jonas/miniconda3/bin/conda run -p /home/jonas/miniconda3/envs/ldsc"
LDSC_MUNGE="${LDSC_ENV} /home/jonas/anders_neededConda/ldsc/munge_sumstats.py"
LDSC="${LDSC_ENV} /home/jonas/anders_neededConda/ldsc/ldsc.py"
rm -rf ~/sysu_day5_ldscore
mkdir -p ~/sysu_day5_ldscore
cd ~/sysu_day5_ldscore
cp -sf ${SHARED_PATH}/online.R .
cp -sf ${SHARED_PATH}/newPlotPlink.R .

cp ${INPUT_PATH}/BBJ*.gz ./
cp ${INPUT_PATH}/w_hm3.snplist ./
ln -sf ${INPUT_PATH}/eas_ldscores ./
ln -sf ${SHARED_PATH}/locuszoom_ref/hg19 ./

ls ./

The **txt.gz** contains the GWAS summary statistics for cholesterol measurements LDL and HDL. LDL is a risk biomarker for cardiovascular diseases while HDL is correlated with reduced risk of cardiovascular disease. The folder **eas_ldscores** containts precalulated LD between SNPs in the east asian populatoin. 

In [ ]:
setwd(path.expand("~/sysu_day5_ldscore"))

### Exploring the summary statistics

In [ ]:
echo number of lines for HDLC:
zcat BBJ_HDLC.txt.gz | wc -l

echo number of lines for LDLC:
zcat BBJ_LDLC.txt.gz | wc -l


Lets look at the first 10 lines of the GWAS summary statistics for HDLC

In [ ]:
zcat BBJ_HDLC.txt.gz |head -n 10 | column -t

In [ ]:
# set up python working space
import os
work_d = os.path.expanduser("~/sysu_day5_ldscore")
os.chdir(work_d)


### visualize the summery stats

Before we perform the LD score regression we visualize the results using R

In [ ]:
#load package for fast reading of data
suppressMessages(require(data.table))
suppressMessages(require(R.utils))
## read positions (hg38)
data <- fread("./BBJ_HDLC.txt.gz", sep="\t", header=TRUE, stringsAsFactors=FALSE, data.table=FALSE)

#print the first 6 lines
head(data)

Let's make a manhattan plot

In [ ]:
source("./online.R")
options(repr.plot.width = 10, repr.plot.height = 6)
manPlot(data$P,chr=as.integer(data$CHR),cap = 10^-300,main="HDLC")

Let's zoom in!

In [ ]:
w <- which.min(data$P)
pos <- data$POS[w]
chr <- data$CHR[w]

win <- 5e4
region <- subset(data,CHR==chr & POS > pos-win &  POS < pos+win)

#truncate p-value
region$P <- pmax(10^-300,region$P)
source("newPlotPlink.R")


#plot
locusZoomNoLD(region$P,chr=chr,pos=region$POS,main="LocusZoom",geneticMap="hg19/genetic_map_GRCh37_chr",refGenes="hg19/refGeneHG19.gz")

 - Based on the plot what is the candidate gene?
 - Can we be sure this is the causal gene?
The lowest p-value is very low and it is hard to see all of the peaks. Lets set the lowest p-value to $10^{-30}$

In [ ]:
manPlot(data$P,chr=as.integer(data$CHR),cap = 10^-30,main="HDLC")

 - how many chromsomes have at least one genome wide significant SNP?

In [ ]:
qqPlot(data$P,cap=1e-300,main="HDLC")
#calculate the inflation factors
X2 <- qchisq(data$P,lower=FALSE,df=1)
lambda <- median(X2)/qchisq(0.5,df=1)
cat("inflaction factors (GC)=",lambda)

 - How does the QQ plot look? Anything we should be worried about?
 - Try to change the cap option above in order to better see the shape of the QQ plot. e.g. cap=1e-30.
 - What does the inflation factor indicate

Lets also see the qq plot for LDLC

In [ ]:
dataLDL <- fread("./BBJ_LDLC.txt.gz", sep="\t", header=TRUE, stringsAsFactors=FALSE, data.table=FALSE)
qqPlot(dataLDL$P,cap=1e-300,main="LDLC")

#calculate the inflation factors
X2 <- qchisq(dataLDL$P,lower=FALSE,df=1)
lambda <- median(X2)/qchisq(0.5,df=1)
cat("inflaction factors (GC)=",lambda)

 - Is LDL cholesterol p-valus inflated?
 
 To understand whether any inflation is due to confounding such as population structure or if it due to a trait being very polygenetic we can perform LD score regression

### Step 1: Munge sumstats

Before the LD score regression analysis, we need to format and clean the raw GWAS summery statistics. We also need to merge the data with the pre-calulated LD measuments. 

In [ ]:
#takes around 1 min
${LDSC_MUNGE} \
    --sumstats ./BBJ_HDLC.txt.gz \
    --merge-alleles ./w_hm3.snplist \
    --a1 ALT \
    --a2 REF \
    --chunksize 500000 \
    --out BBJ_HDLC


 - How many SNPs were merged with the pre calculated LD measurements?
 - The inflation factor (Lambda GC) is slightly different then what we calculated in R. why? 
 
 Lets also run the merging for LDLC

In [ ]:

${LDSC_MUNGE} \
    --sumstats ./BBJ_LDLC.txt.gz \
    --merge-alleles w_hm3.snplist \
    --a1 ALT \
    --a2 REF \
    --chunksize 500000 \
    --out BBJ_LDLC


After munging, you will get two munged and formatted files that can be matched with the LD reference data: 

In [ ]:
zcat BBJ_HDLC.sumstats.gz | head -n2
zcat BBJ_LDLC.sumstats.gz | head -n2
echo ---- number of lines ----
zcat BBJ_HDLC.sumstats.gz | wc -l
zcat BBJ_LDLC.sumstats.gz | wc -l

### LD score regression
Univariate LD score regression is utilized to estimate heritbility 
and confuding factors (cryptic relateness and population stratification)
 of a certain trait.
 
 Reference : Bulik-Sullivan, Brendan K., et al. "LD Score regression 
distinguishes confounding from polygenicity in genome-wide association 
studies." Nature genetics 47.3 (2015): 291-295.

Using the munged sumstats, we can run the LD score regression:

In [ ]:

${LDSC} \
  --h2 BBJ_HDLC.sumstats.gz \
  --ref-ld-chr eas_ldscores/ \
  --w-ld-chr eas_ldscores/ \
  --out BBJ_HDLC


In [ ]:

${LDSC} \
  --h2 BBJ_LDLC.sumstats.gz \
  --ref-ld-chr eas_ldscores/ \
  --w-ld-chr eas_ldscores/ \
  --out BBJ_LDLC

In [ ]:
# Lest's check the results for HDLC:
cat BBJ_HDLC.log


We can see that from the log:
 - Observed scale h2 = 0.1583
 - lambda GC = 1.1523
 - intercept = 1.0563
 - Ratio = 0.1981

The ratio measures the proportion of the inflation in the mean chi^2 that the LD Score regression that is explained by causes other than polygenic heritability i.e. confounding. The value of ratio
 should be close to zero, though in practice values of 10-20% are not uncommon.